# 🐖 00 | Descarga y verificación del dataset

**RECURSO REQUERIDO: CPU (sin acelerador)**. Este notebook realiza exclusivamente su propia etapa. **No cambies CPU/GPU dentro del notebook.**

Proyecto: Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖. Datos: FaroPigSeg, licencia reportada CC BY-NC 4.0; citar fuente https://github.com/yudong888/FaroPig-Datasets.

**Ruta general:** `MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖/`. Ejecutar en orden 00 → 01 → 02 → 03 → 04. Todos los resultados se guardan en Drive.

## Objetivo
Descargar **solo si el dataset no está ya extraído**. Comprobar que contiene imágenes y etiquetas originales. No se calculan métricas y no se entrena ningún modelo.

Si el dataset ya existe en Drive, esta etapa finaliza en pocos segundos, sin descargarlo otra vez.

In [ ]:
%pip -q install requests tqdm

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, os

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError(
        'No se encontró la carpeta principal. Verifica la cuenta de Google y el nombre de la carpeta: '
        'Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖'
    )
DATA_ROOT = ROOT / '01_Datasets' / 'FaroPigSeg' / 'extraido' / 'FaroPigSeg'
DATASET_DIR = ROOT / '01_Datasets' / 'FaroPigSeg'
RES = ROOT / '03_Resultados'
EXPO = ROOT / '05_Exportaciones_ZIP'
for sub in ('01_Exploracion','02_Segmentacion','03_Metricas','04_Figuras','05_Reportes','06_Entrenamiento'):
    (RES / sub).mkdir(parents=True, exist_ok=True)
EXPO.mkdir(parents=True,exist_ok=True)
print('Proyecto:',ROOT)

In [ ]:
import hashlib, zipfile, requests, shutil, stat
from tqdm.auto import tqdm
SOURCE='https://data.chalearnlap.cvc.uab.cat/FaroPig/FaroPigSeg.zip'
ZIP=DATASET_DIR/'FaroPigSeg_original.zip'

def estructura_completa(path):
    return all((path/s/k).is_dir() and any((path/s/k).iterdir())
               for s in ('train','val','test') for k in ('images','labels'))

if estructura_completa(DATA_ROOT):
    print('✅ FaroPigSeg ya está extraído en Drive. No se descarga ni se modifica.')
else:
    DATASET_DIR.mkdir(parents=True,exist_ok=True)
    if not (ZIP.is_file() and ZIP.stat().st_size>10_000_000 and zipfile.is_zipfile(ZIP)):
        PART=Path('/content/FaroPigSeg.zip.part')
        with requests.get(SOURCE,stream=True,timeout=(30,180)) as req:
            req.raise_for_status()
            if 'html' in req.headers.get('Content-Type','').lower():
                raise RuntimeError('El servidor devolvió HTML en vez del dataset.')
            with open(PART,'wb') as f:
                for block in tqdm(req.iter_content(chunk_size=1024*1024),desc='Descarga FaroPigSeg'):
                    if block:f.write(block)
        if PART.stat().st_size<10_000_000 or not zipfile.is_zipfile(PART):
            raise RuntimeError('La descarga no es un ZIP válido.')
        shutil.copy2(PART,ZIP)
    with zipfile.ZipFile(ZIP) as z:
        if z.testzip(): raise RuntimeError('ZIP dañado, revisar la descarga.')
        target=DATASET_DIR/'extraido'
        for entry in z.infolist():
            dest=(target/entry.filename).resolve()
            if not dest.is_relative_to(target.resolve()):
                raise RuntimeError('Ruta insegura en el ZIP.')
            if stat.S_ISLNK(entry.external_attr>>16):
                raise RuntimeError('Enlace simbólico no permitido en ZIP.')
        z.extractall(target)
    if not estructura_completa(DATA_ROOT):
        raise RuntimeError(f'ZIP extraído, pero la estructura no coincide con: {DATA_ROOT}')

counts={s:{k:len(list((DATA_ROOT/s/k).iterdir())) for k in ('images','labels')}
        for s in ('train','val','test')}
print('Archivos por partición:',counts)
report={'etapa':'00_descarga_CPU','fecha_utc':datetime.now(timezone.utc).isoformat(),
        'source':SOURCE,'dataset':str(DATA_ROOT),'conteos':counts}
dest=RES/'05_Reportes'/'CERDOS_00_Descarga_manifest.json'
dest.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf8')
print('✅ Registro guardado:',dest)

**Siguiente:** ejecutar `01_CERDOS_Auditoria_CPU.ipynb` con **CPU**. No actives GPU aún.